### The actual training of the model

In [7]:
import torch
from transformers import RobertaForSequenceClassification, Trainer, TrainingArguments
import datasets
from sklearn.metrics import precision_recall_fscore_support, accuracy_score
import numpy as np
import os
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from transformers import RobertaTokenizer, RobertaForSequenceClassification,  Trainer, TrainingArguments, DataCollatorWithPadding

In [2]:
labels = [ "Self-direction: thought", "Self-direction: action", "Stimulation",  "Hedonism", "Achievement", "Power: dominance", "Power: resources", "Face", "Security: personal", "Security: societal", "Tradition", "Conformity: rules", "Conformity: interpersonal", "Humility", "Benevolence: caring", "Benevolence: dependability", "Universalism: concern", "Universalism: nature", "Universalism: tolerance", "No Value"]
num_labels = len(labels)

train_dataset = datasets.load_from_disk("../datasets/processed_train")
validation_dataset = datasets.load_from_disk("../datasets/processed_validation")

In [3]:
model_name = "roberta-base"
tokenizer = RobertaTokenizer.from_pretrained(model_name)
model = RobertaForSequenceClassification.from_pretrained("roberta-base", num_labels=num_labels)

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [4]:
class CustomRobertaForSequenceClassification(RobertaForSequenceClassification):
    def forward(self, input_ids=None, attention_mask=None, labels=None):
        outputs = self.roberta(input_ids, attention_mask=attention_mask)
        sequence_logits = outputs[0]  # Shape: [batch_size, sequence_length, num_labels]
        cls_logits = sequence_logits[:, 0, :]  # Shape: [batch_size, num_labels]
        logits = self.classifier(cls_logits)
        if labels is not None:
            # loss_fct = CustomLossWithAtLeastOnePositive(nn.BCEWithLogitsLoss())
            loss_fct = nn.BCEWithLogitsLoss()
            loss = loss_fct(logits, labels.float())
            return (loss, logits)
        return (logits,)

model = CustomRobertaForSequenceClassification.from_pretrained(model_name, num_labels=num_labels)

# Freezing the weights
# for param in model.roberta.parameters():
#     param.requires_grad = False

# Adding a final classification layer
model.classifier = nn.Sequential(
    nn.ReLU(),
    nn.Dropout(0.3),
    nn.Linear(model.config.hidden_size, num_labels),
)

Some weights of CustomRobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [5]:
def compute_metrics(pred):
    logits, labels = pred
    probs = torch.sigmoid(torch.tensor(logits))
    
    thresholds = [0.1, 0.2, 0.3, 0.4, 0.5]
    best_threshold, best_f1 = 0, 0
    for t in thresholds:
        prediction = (probs >= t).int()
        _, _, f1, _ = precision_recall_fscore_support(labels, prediction, average='samples')
        if f1 > best_f1:
            best_threshold, best_f1 = t, f1
    
    prediction = (probs >= best_threshold).int()
    max_probs = probs.argmax(dim=1)
    for i in range(prediction.shape[0]):
        if prediction[i].sum() == 0:  # No label selected
            prediction[i, max_probs[i]] = 1  # Assign the label with max probability

    
    # Calcul des métriques pour un problème multi-label
    precision, recall, f1, _ = precision_recall_fscore_support(labels, prediction, average='samples',zero_division=0)
    accuracy = accuracy_score(labels, prediction)
    
    return {
        'accuracy': accuracy,
        'f1': f1,
        'precision': precision,
        'recall': recall
    }

In [8]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [9]:
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=4,             
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=2e-5,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    save_total_limit=2,
    warmup_ratio=0.2
)

/Users/edabier/miniconda3/envs/roberta_env/lib/python3.11/site-packages/transformers/training_args.py:1545: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [10]:
optimizer = torch.optim.AdamW(
    model.parameters(),  # Parameters to optimize
    lr=2e-5,             # Learning rate
    weight_decay=0.01    # Weight decay for regularization
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=validation_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    optimizers=(optimizer, None)
)

In [11]:
trainer.train()

  0%|          | 0/22380 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
trainer.save_model("./results")
tokenizer.save_pretrained("./results")